# Clusters unmixing experiments

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dnevo/clusters_unmixing/blob/main/notebooks/00_clusters_unmixing_experiments.ipynb)

This notebook runs the configured spectral unmixing experiments and presents the results in a notebook-friendly review format.

It covers:
- loading the project configuration and executing the configured experiment runs
- inspecting raw and normalized cluster spectra for each run
- comparing cosine off-diagonal statistics across preprocessing steps
- reviewing model metrics for the selected unmixing methods
- previewing predicted abundances against the synthetic ground truth
- visualizing synthetic pixel spectra generated for each run


In [1]:
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    project_root = Path("/content/clusters_unmixing")
    if not project_root.exists():
        subprocess.check_call([
            "git",
            "clone",
            "https://github.com/dnevo/clusters_unmixing.git",
            str(project_root),
        ])
else:
    NOTEBOOK_DIR = Path.cwd()
    project_root = NOTEBOOK_DIR.parent

SRC_DIR = project_root / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [2]:
import torch

if IN_COLAB and torch.cuda.is_available():
    # Optional: accelerates the mamba model's selective scan with a fused CUDA
    # kernel instead of the pure-PyTorch fallback loop. Skipped entirely off of
    # a GPU runtime; any failure below just falls back to the pure-PyTorch scan
    # rather than breaking the run.
    def _find_prebuilt_wheel(repo: str, package_prefix: str) -> str | None:
        # mamba-ssm/causal-conv1d publish prebuilt wheels as GitHub release
        # assets, named for a specific torch/CUDA/Python/C++-ABI combo. Building
        # from source instead takes several minutes and needs --no-build-isolation
        # (their setup.py imports torch while compiling, which pip's isolated
        # build env doesn't have) - so try to find a matching wheel first.
        import json
        import re
        import urllib.request

        cuda_version = (torch.version.cuda or "").replace(".", "")
        if not cuda_version:
            return None
        cuda_tag = re.escape(f"cu{cuda_version}")
        torch_tag = re.escape("torch" + ".".join(torch.__version__.split("+")[0].split(".")[:2]))
        abi_tag = r"cxx11abi(?:True|False|TRUE|FALSE)"
        py_tag = re.escape(f"cp{sys.version_info.major}{sys.version_info.minor}")

        pattern = re.compile(
            rf"^{re.escape(package_prefix)}-[\d.]+\+{cuda_tag}{torch_tag}{abi_tag}"
            rf"-{py_tag}-{py_tag}-linux_x86_64\.whl$"
        )
        try:
            with urllib.request.urlopen(
                f"https://api.github.com/repos/{repo}/releases/latest", timeout=15
            ) as resp:
                release = json.load(resp)
        except Exception:
            return None

        for asset in release.get("assets", []):
            if pattern.match(asset["name"]):
                return asset["browser_download_url"]
        return None

    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "packaging", "ninja"])

        conv1d_url = _find_prebuilt_wheel("Dao-AILab/causal-conv1d", "causal_conv1d")
        mamba_url = _find_prebuilt_wheel("state-spaces/mamba", "mamba_ssm")

        if conv1d_url and mamba_url:
            print("Found matching prebuilt wheels; installing without compiling.")
            subprocess.check_call([sys.executable, "-m", "pip", "install", conv1d_url])
            subprocess.check_call([sys.executable, "-m", "pip", "install", mamba_url])
        else:
            print("No matching prebuilt wheel found for this torch/CUDA/Python combo; building from source (slower).")
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--no-build-isolation", "causal-conv1d"]
            )
            subprocess.check_call(
                [sys.executable, "-m", "pip", "install", "--no-build-isolation", "mamba-ssm"]
            )
    except subprocess.CalledProcessError as exc:
        print(f"mamba-ssm install failed ({exc}); falling back to the pure-PyTorch scan.")


In [3]:
from clusters_unmixing.utils import run_experiments_notebook

In [4]:
run_experiments_notebook(project_root=project_root)
if IN_COLAB:
    # disconnect from the runtime to avoid timeout
    from google.colab import runtime  # type: ignore[import-not-found]
    runtime.unassign()


--- Run 1/2 | model=small_mlp ---


Epoch    1 | train=5.168123e-02 | val=4.276506e-02 | val_abund=4.255401e-02 | val_recon=2.110483e-03


Epoch   20 | train=2.749026e-02 | val=3.128348e-02 | val_abund=3.108162e-02 | val_recon=2.018555e-03


Epoch   40 | train=2.452828e-02 | val=3.021227e-02 | val_abund=3.001110e-02 | val_recon=2.011706e-03


--- Run 1/2 | model=kan ---


...

### Model metrics

model,sunsal,vpgdu,small_mlp
metric,,,
abundance_rmse,0.132537,0.140076,0.137882
reconstruction_rmse,0.044620,0.044841,0.045299


### Abundances

pixel_index,source,abundance_rmse,reconstruction_rmse,endmember_1,endmember_2,endmember_3,endmember_4,endmember_5,endmember_6
1969,true,0.000000,0.044237,0.000000,0.200000,0.160000,0.060000,0.240000,0.340000
1969,small_mlp,0.074821,0.044231,0.119073,0.076877,0.200646,0.081686,0.196182,0.325536
1969,sunsal,0.195191,0.044179,0.091938,-0.001175,0.494569,-0.002726,-0.000334,0.417728
1969,vpgdu,0.204965,0.044243,0.057910,0.000000,0.541042,0.000000,0.002360,0.398688
3822,true,0.000000,0.044383,0.080000,0.140000,0.180000,0.120000,0.000000,0.480000
3822,small_mlp,0.090035,0.044361,0.072513,0.080079,0.192163,0.155358,0.157244,0.342643
3822,sunsal,0.093632,0.044352,0.091026,-0.000004,0.306631,0.212939,-0.000002,0.389411
3822,vpgdu,0.096104,0.044353,0.092067,0.000000,0.323764,0.171329,0.037528,0.375312
3937,true,0.000000,0.044520,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000
3937,small_mlp,0.010271,0.044605,0.000010,0.005907,0.008215,0.007082,0.000692,0.978094


### Synthetic pixel spectra preview